<a href="https://colab.research.google.com/github/abilashkannanv/AIML/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = "voc-262423617156170395256969020d2c765580.35946547"

In [19]:
!pip install langchain-openai
from langchain_openai import ChatOpenAI

In [3]:
!pip install langchain-community
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

In [4]:
import os
!pip install pypdf
from pypdf import PdfReader

pdf_names = ['AIML.pdf', 'AIMLDL.pdf', 'ML1.pdf', 'ML.pdf', 'Chunking.pdf', 'RAG.pdf']
base_directory = '/content/'

raw_documents = []
for pdf_file in pdf_names:
    file_path = os.path.join(base_directory, pdf_file)
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        for doc in docs:
            doc.metadata['doc_title'] = pdf_file.replace('.pdf', '')
        raw_documents.extend(docs)
        print(f"Successfully loaded {pdf_file}")
    except Exception as e:
        print(f"Error loading {pdf_file}: {e}")

Successfully loaded AIML.pdf
Successfully loaded AIMLDL.pdf
Successfully loaded ML1.pdf
Successfully loaded ML.pdf
Successfully loaded Chunking.pdf
Successfully loaded RAG.pdf


Now, let's chunk the documents using `RecursiveCharacterTextSplitter` with the specified `chunk_size` and `chunk_overlap`.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 600
chunk_overlap = int(chunk_size * 0.15) # 15% overlap

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(raw_documents)

# Add 1-indexed page number and chunk_id to metadata
for i, chunk in enumerate(chunks):
    if 'page' in chunk.metadata:
        chunk.metadata['page'] = chunk.metadata['page'] + 1 # Convert to 1-indexed page
    else:
        chunk.metadata['page'] = 'N/A' # Handle cases where page metadata might be missing
    chunk.metadata['chunk_id'] = i

print(f"Created {len(chunks)} chunks.")
print("First chunk metadata:")
display(chunks[0].metadata)
print("First chunk content preview:")
display(chunks[0].page_content[:200] + "...")

Created 4455 chunks.
First chunk metadata:


{'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)',
 'creator': 'Microsoft® Word 2016',
 'creationdate': '2022-08-12T08:58:55+00:00',
 'author': 'cvrpexam',
 'moddate': '2024-06-12T14:26:27+00:00',
 'source': '/content/AIML.pdf',
 'total_pages': 93,
 'page': 1,
 'page_label': '1',
 'doc_title': 'AIML',
 'start_index': 0,
 'chunk_id': 0}

First chunk content preview:


'1 \nSTUDY MATERIAL \n \n \nOn  \n \n \nArtificial intelligence \nand Machine Learning \n \n(For 6th Semester CSE,CVRP) \n \n \n \n \n \n \n \n \nPrepared by : \n \n PRANGYA PARAMITA MOHAPATRA,ASST.PROF.(CSE),CVRP...'

### 1. Initialize Embeddings and Vector Store

First, we'll initialize our embedding model. We'll use `OpenAIEmbeddings` for this purpose. Then, we'll create a FAISS vector store from our document `chunks` using these embeddings. FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors.

In [19]:
!pip install faiss-cpu

from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.vectorstores import FAISS

# Initialize the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key = "voc-2033465586156170068dbe3820ba1c1.93790491",
                    openai_api_base = "https://openai.vocareum.com/v1")

# Create a FAISS vector store from the chunks
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"FAISS vector store created with {vectorstore.index.ntotal} documents.")

FAISS vector store created with 4455 documents.


### 2. Initialize Retriever

Next, we'll create a retriever from our FAISS vector store. The retriever's job is to fetch relevant documents based on a given query.

In [20]:
retriever = vectorstore.as_retriever()
print("Retriever initialized.")

Retriever initialized.


### 3. Define Prompt Template

Now, we'll define a prompt template that will guide the LLM to provide accurate and cited answers based on the retrieved context.

In [22]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks. \n
Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. \n
Use three sentences maximum and keep the answer concise.\n
Question: {question} \n
Context: {context} \n
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

print("Prompt template defined.")

Prompt template defined.


### 4.1. Prompt Template Variant P1 (Concise)

This prompt template (`prompt_p1`) is designed for concise answers (<= 120 tokens), strictly using the provided context, and including citations in the format `[doc_title#page]`.

In [29]:
from langchain_core.prompts import ChatPromptTemplate

template_p1 = """You are an assistant for question-answering tasks. \n
Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. \n
Answer in <= 120 tokens. Provide citations in the format [doc_title#page] for each piece of information.

Question: {question} \n
Context: {context} \n
Answer:"""

prompt_p1 = ChatPromptTemplate.from_template(template_p1)

print("Prompt template P1 (concise) defined.")

Prompt template P1 (concise) defined.


### 4.2. Prompt Template Variant P2 (Reasoned)

This prompt template (`prompt_p2`) extends P1 by asking the LLM to first list relevant evidence snippets and then provide its concise answer, maintaining the token limit and citation format.

In [30]:
from langchain_core.prompts import ChatPromptTemplate

template_p2 = """You are an assistant for question-answering tasks. \n
Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. \n
Answer in <= 120 tokens. Provide citations in the format [doc_title#page] for each piece of information.
First, briefly list the 2 most relevant supporting snippets from the context under an "Evidence:" bullet point. Then, provide your concise answer under an "Answer:" bullet point.

Question: {question} \n
Context: {context} \n
Answer:"""

prompt_p2 = ChatPromptTemplate.from_template(template_p2)

print("Prompt template P2 (reasoned) defined.")

Prompt template P2 (reasoned) defined.


### 4. Construct RAG Chain

Finally, we'll assemble all the components: the retriever, the LLM, and the prompt template, into a RAG chain using LangChain's expression language. This chain will take a question, retrieve relevant documents, format them into the prompt, and then pass the prompt to the LLM to generate an answer.

In [25]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = "voc-2033465586156170068dbe3820ba1c1.93790491"

from langchain_openai import ChatOpenAI

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

my_llm = ChatOpenAI(model_name = "gpt-4o-mini", api_key = "voc-2033465586156170068dbe3820ba1c1.93790491",
                    base_url = "https://openai.vocareum.com/v1")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | my_llm
    | StrOutputParser()
)

print("RAG chain constructed.")

RAG chain constructed.


### 5. Generate Response

Now you can ask a question and get a generated answer from the RAG chain.

In [26]:
question = "What are some key aspects of Chunking?"
response = rag_chain.invoke(question)
print(response)

Key aspects of chunking include breaking down large texts into smaller, manageable segments called chunks, which optimizes content relevance for storage in vector databases. It involves finding a balance between chunk size—large enough to convey meaningful information but small enough for efficient application performance. There are various chunking methods, such as fixed-size chunking, but approaches must be tailored to specific use cases as one solution may not fit all.
